# 03B — SLICES Domain-Adaptive Pretraining: MSI full run

> **ENTRENAMIENTO CIENTÍFICO COMPLETO — requiere una asignación MSI con GPU NVIDIA/CUDA.**  
> Este notebook documenta y permite inspeccionar la configuración y los resultados. El job pesado se ejecuta fuera de Jupyter mediante `dapt/scripts/run_dapt_msi.py`, solicitado explícitamente con SLURM. Abrir o ejecutar este notebook no envía un job.

## Pregunta científica

¿El continued pretraining de ChemBERTa mediante masked language modeling sobre cadenas SLICES mejora la predicción de tokens estructurales en máscaras fijas de validation y holdout? La comparación justa es PRE-DAPT versus POST-DAPT usando el mismo modelo base, tokenizer SLICES adaptado, vocabulario y máscaras. No se incluyen aquí regresión de energía, clasificación, ranking de polimorfos ni fine-tuning downstream.

03A responde “¿funciona la implementación?”; este análisis MSI pregunta “¿se adaptó ChemBERTa al dominio SLICES?”. La lógica científica —parser, categorías, split, chunking, masks, métricas y plots— se importa del mismo `dapt/scripts/dapt_utils.py` usado por 03A.


## 1. Entorno, versiones y ubicación del proyecto

El runner exige CUDA y detiene el entrenamiento si no está disponible; no hace fallback silencioso a CPU. En este notebook se puede inspeccionar la configuración incluso desde una Mac, pero esa sesión no es válida para entrenar el corpus completo.

In [ ]:
from pathlib import Path
import json
import sys
import yaml
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "dapt").is_dir() and (PROJECT_ROOT / "audit").is_dir()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("No se encontró la raíz del proyecto Transformers.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "dapt/scripts"))
from dapt_utils import generate_scientific_plots, read_table

CONFIG_PATH = PROJECT_ROOT / "dapt/configs/dapt_msi.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA build version:", torch.version.cuda)
print("MSI config:", CONFIG_PATH)
display(pd.Series(config, name="configured value").to_frame())


## 2. Configuración reproducible del entrenamiento completo

`dapt_msi.yaml` es la fuente editable de hiperparámetros: `max_structures: null` obliga a usar todo el dataset, hasta 8 epochs, batches 16/32, acumulación 4 (batch efectivo nominal 64), AdamW, learning rate 5e-5, weight decay .01, warmup lineal 10%, máscara MLM .15, clipping 1.0 y early stopping con paciencia 2. Si ocurre CUDA OOM, el runner informa sugerencias (8×8 o 4×16) y falla; nunca cambia los batches silenciosamente.

In [ ]:
assert config.get("max_structures") is None, "MSI full run must not use the Mac smoke-test cap."
print("Effective train batch:", config["train_batch_size"] * config["gradient_accumulation_steps"])
print("Configured epochs (maximum):", config["epochs"])
print("MLM probability:", config["mlm_probability"])
print("Full-corpus mode:", config["mode"] == "full")


## 3. Hardware MSI y regla de lanzamiento

El archivo `submit_dapt_msi.slurm` deja marcadores para confirmar con MSI `account`, `partition`, petición GPU y tiempo máximo. No se fija ningún account privado. El SLURM activa un entorno mediante `CONDA_ENV_NAME`, imprime nodo/fecha/Python/`nvidia-smi` y ejecuta el script independiente. Para reanudar intenta encontrar el último checkpoint existente.

**No ejecutar `subprocess`, `sbatch` ni iniciar entrenamiento desde esta celda.** La línea de abajo solo muestra el comando que se usará después de verificar la configuración del cluster.

In [ ]:
if sys.platform == "darwin":
    cuda_available_here = False
    print("macOS inspection session: CUDA was not queried; full DAPT belongs on MSI.")
else:
    cuda_available_here = torch.cuda.is_available()
    print("CUDA available in this current kernel:", cuda_available_here)
    if cuda_available_here:
        print("GPU count:", torch.cuda.device_count())
        print("GPU model:", torch.cuda.get_device_name(0))
        print("PyTorch CUDA version:", torch.version.cuda)
        try:
            print("GPU VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
        except Exception as exc:
            print("Could not read VRAM metadata:", exc)
    else:
        print("This is an inspection-only session. Full DAPT stops without an MSI CUDA allocation.")

SLURM_COMMAND = "sbatch dapt/scripts/submit_dapt_msi.slurm"
print("Future explicit launch command (not executed):", SLURM_COMMAND)


## 4. Qué hace `run_dapt_msi.py`

1. Lee YAML y rechaza `max_structures` distinto de `null`. 2. Exige CUDA antes de cargar corpus/modelo. 3. Encuentra el CSV limpio y tokenizer adaptado. 4. Crea o reutiliza el split persistente por `composition_key`. 5. Comprueba fidelidad lexical mínima 99.5% sobre ≥5,000 estructuras y verifica lexemas únicos. 6. Trocea solo grupos completos de aristas tras asignar split. 7. Crea fixed masks deterministas y evalúa PRE. 8. Redimensiona embeddings usando inicialización estándar, comprobando la preservación del vocabulario original. 9. Entrena todos los parámetros del encoder/MLM con AdamW y scheduler lineal. 10. Guarda checkpoints, best checkpoint, métricas y figuras; `--resume auto` encuentra el run reanudable más reciente. 11. Solo publica el modelo canónico si pasan las verificaciones críticas.

In [ ]:
RUNNER = PROJECT_ROOT / "dapt/scripts/run_dapt_msi.py"
SLURM_FILE = PROJECT_ROOT / "dapt/scripts/submit_dapt_msi.slurm"
print("Runner:", RUNNER)
print("SLURM template:", SLURM_FILE)
print("Manual launch command:")
print("  sbatch dapt/scripts/submit_dapt_msi.slurm")
print("No job was submitted from this notebook.")


## 5. Seleccionar e inspeccionar un run completado

Cada lanzamiento crea `dapt/outputs/msi/msi_dapt_YYYYMMDD_HHMMSS/`; una reanudación conserva la carpeta original. Selecciona una carpeta específica abajo para que el análisis sea reproducible. Si se deja como `None`, el notebook toma la carpeta MSI más recientemente modificada que contenga el resumen final.

In [ ]:
MSI_OUTPUT_ROOT = PROJECT_ROOT / "dapt/outputs/msi"
RUN_DIR_OVERRIDE = None  # Por ejemplo: "dapt/outputs/msi/msi_dapt_20261005_120000"
if RUN_DIR_OVERRIDE:
    RUN_DIR = (PROJECT_ROOT / RUN_DIR_OVERRIDE).resolve()
else:
    candidates = [p for p in MSI_OUTPUT_ROOT.glob("msi_dapt_*") if (p / "dapt_run_summary.json").is_file()]
    RUN_DIR = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

if RUN_DIR is None:
    print("Todavía no hay resultados MSI para inspeccionar. El notebook no inicia el entrenamiento.")
else:
    print("Selected full-run results:", RUN_DIR)
    run_summary = json.loads((RUN_DIR / "dapt_run_summary.json").read_text(encoding="utf-8"))
    display(pd.Series({
        "device": run_summary.get("device", {}).get("device"),
        "precision": run_summary.get("precision"),
        "structures": run_summary.get("n_structures"),
        "chunks": run_summary.get("n_chunks"),
        "epochs completed": run_summary.get("epochs_completed"),
        "decision": run_summary.get("decision"),
        "accepted": run_summary.get("accepted"),
    }, name="run summary").to_frame())


## 6. Auditoría PRE/POST con posiciones fijas

El runner evalúa validation y holdout con el mismo conjunto de posiciones ocultas antes y después del entrenamiento. La tabla global incluye cross-entropy, perplexity, Top-1/3/5, MRR y probabilidad media del token verdadero. El holdout queda aislado de la optimización y del early stopping.

In [ ]:
if RUN_DIR is not None:
    metric_rows = []
    for split in ["validation", "holdout"]:
        for stage in ["pre", "post"]:
            path = RUN_DIR / f"{stage}_dapt_{split}_metrics.json"
            if path.exists():
                values = json.loads(path.read_text(encoding="utf-8"))
                metric_rows.append({"split": split, "stage": stage.upper(), **values})
    metrics_df = pd.DataFrame(metric_rows)
    display(metrics_df)
    if (RUN_DIR / "pre_vs_post_dapt_metrics.csv").exists():
        display(pd.read_csv(RUN_DIR / "pre_vs_post_dapt_metrics.csv"))


## 7. Aprendizaje por roles gramaticales y vocabulario

Los archivos por categoría comparan `space_group`, `element`, `node_index`, `periodic_vector` y `other`. Tablas adicionales separan tokens originales de tokens SLICES añadidos, listan resultados por elemento/vector/índice y cuantifican movimiento de embeddings (norma, desplazamiento L2 relativo y coseno). Un mayor movimiento no se interpreta automáticamente como mejor calidad.

In [ ]:
if RUN_DIR is not None:
    for filename, title in [
        ("pre_vs_post_metrics_by_category.csv", "PRE/POST por rol SLICES"),
        ("original_vs_added_token_metrics.csv", "Vocabulario original vs añadido"),
        ("element_mlm_metrics.csv", "Métricas por elemento"),
        ("periodic_vector_mlm_metrics.csv", "Métricas por vector periódico"),
        ("node_index_mlm_metrics.csv", "Métricas por rango de índice"),
        ("embedding_shift.csv", "Desplazamiento de embeddings"),
    ]:
        path = RUN_DIR / filename
        if path.exists():
            print(title)
            table = pd.read_csv(path)
            display(table.head(20))


## 8. Figuras científicas

La función compartida genera las 20 figuras definidas para el análisis: curvas, métricas PRE/POST, comparación original/añadido, detalle por elemento y vector, ejemplos, matrices de confusión, chunking, movimiento de embeddings, PCA ilustrativa y comparación lexical de tokenizers. La figura lexical del tokenizer ZINC es una ilustración de segmentación, **no** una comparación de MLM loss entre tokenizers.

In [ ]:
if RUN_DIR is not None:
    expected = [
        "01_mlm_training_curves.png", "02_pre_vs_post_overall_mlm.png",
        "03_pre_vs_post_perplexity.png", "04_accuracy_by_slices_category.png",
        "05_loss_by_slices_category.png", "06_top5_accuracy_by_category.png",
        "07_original_vs_added_vocab_accuracy.png", "08_periodic_vector_learning.png",
        "09_element_mlm_learning.png", "10_true_token_probability_shift.png",
        "11_probability_gain_by_token_type.png", "12_embedding_cosine_change.png",
        "13_embedding_displacement.png", "14_selected_embeddings_pca.png",
        "15_masked_prediction_examples.png", "16_periodic_vector_confusion_post_dapt.png",
        "17_element_confusion_post_dapt.png", "18_dapt_chunking_statistics.png",
        "19_dapt_summary_dashboard.png", "20_original_vs_adapted_tokenization.png",
    ]
    missing = [name for name in expected if not (RUN_DIR / name).exists()]
    if missing:
        print("Generating/replacing scientific PNGs from this run's saved artifacts...")
        generated = generate_scientific_plots(
            RUN_DIR,
            tokenizer_path=PROJECT_ROOT / config["tokenizer_path"],
            base_model_name=config["base_model_name"],
        )
        print("Generated plots:", len(generated))
    present = [name for name in expected if (RUN_DIR / name).exists()]
    print(f"Scientific plots present: {len(present)}/{len(expected)}")
    for name in present:
        print(" -", name)


## 9. Predicciones en ejemplos y checks de aceptación

Inspecciona ejemplos de tokens enmascarados y el top-5 PRE/POST. Los checks críticos implementados por el runner son: pérdidas finitas, split sin fuga, mejora de loss de validation y holdout, tamaño embedding/vocabulario coincidente, fidelidad gramatical, embeddings originales preservados y recarga de checkpoint. La decisión `ACCEPT / REVIEW / REJECT` se guarda con el manifiesto del run; los indicadores deseables se reportan aparte.

In [ ]:
if RUN_DIR is not None:
    examples_file = RUN_DIR / "masked_prediction_examples.csv"
    if examples_file.exists():
        example_df = pd.read_csv(examples_file)
        display(example_df.head(10))
    summary_path = RUN_DIR / "dapt_run_summary.json"
    if summary_path.exists():
        checks = json.loads(summary_path.read_text(encoding="utf-8"))
        print("Decision:", checks.get("decision"))
        print("Critical checks:")
        display(pd.Series(checks.get("critical_checks", {}), name="passed").to_frame())
        print("Desirable checks:")
        display(pd.Series(checks.get("desirable_checks", {}), name="improved").to_frame())


## 10. Resumen final tabular del run MSI

Este resumen reúne hardware, tamaños del corpus, chunking, mejor época y métricas PRE/POST globales y por familia de tokens. El rótulo ACCEPT/REVIEW/REJECT proviene de los checks críticos guardados por el runner; los indicadores deseables se muestran por separado y no sustituyen los checks críticos.

In [ ]:
if RUN_DIR is not None:
    split_info = json.loads((RUN_DIR / "split_summary.json").read_text(encoding="utf-8"))
    counts = split_info["counts"]
    chunk_stats = pd.read_csv(RUN_DIR / "chunking_statistics.csv").iloc[0]
    env = json.loads((RUN_DIR / "environment.json").read_text(encoding="utf-8"))
    best_state_path = RUN_DIR / "best_checkpoint/best_state.json"
    best_epoch = json.loads(best_state_path.read_text()).get("epoch") if best_state_path.exists() else None

    print("=" * 70)
    print("SLICES DAPT — MSI FULL RUN")
    print("=" * 70)
    print("Hardware")
    print("  GPU:", env.get("gpu_name"))
    print("  GPU count:", env.get("gpu_count"))
    print("  VRAM bytes:", env.get("gpu_vram_bytes"))
    print("  CUDA / PyTorch CUDA:", env.get("cuda_version"))
    print("  Precision:", run_summary.get("precision"))
    print("Corpus")
    print("  Structures:", f"{run_summary.get('n_structures', 0):,}")
    for split_name in ["train", "validation", "holdout"]:
        item = counts.get(split_name, {})
        print(f"  {split_name.title():12s} {item.get('structures', 0):,} structures | {item.get('compositions', 0):,} compositions | {item.get('chunks', 0):,} chunks")
    print("  Total chunks:", f"{run_summary.get('n_chunks', 0):,}")
    print("  Structures chunked:", f"{chunk_stats['fraction_structures_chunked']:.2%}")
    print("  Edges preserved:", f"{chunk_stats['fraction_edges_preserved_all_selected']:.2%}")
    print("Training")
    print("  Epochs completed:", run_summary.get("epochs_completed"))
    print("  Best epoch:", best_epoch)

    rows = []
    for split_name in ["validation", "holdout"]:
        before = run_summary.get("pre_dapt", {}).get(split_name, {})
        after = run_summary.get("post_dapt", {}).get(split_name, {})
        for label, key in [("loss", "loss"), ("perplexity", "perplexity"), ("Top-1", "top1_accuracy"), ("Top-5", "top5_accuracy")]:
            rows.append({"split": split_name, "metric": label, "PRE": before.get(key), "POST": after.get(key)})
    category_path = RUN_DIR / "pre_vs_post_metrics_by_category.csv"
    if category_path.exists():
        cat = pd.read_csv(category_path)
        val_cat = cat.loc[cat["split"] == "validation"]
        for category in ["element", "periodic_vector", "node_index", "space_group"]:
            item = val_cat.loc[val_cat["token_category"] == category]
            if len(item):
                item = item.iloc[0]
                rows.append({"split": "validation", "metric": f"{category} Top-1", "PRE": item.get("pre_top1_accuracy"), "POST": item.get("post_top1_accuracy")})
    vocab_path = RUN_DIR / "original_vs_added_token_metrics.csv"
    if vocab_path.exists():
        vocab = pd.read_csv(vocab_path)
        for group in ["original_vocab", "added_slices_vocab"]:
            item = vocab.loc[vocab["original_or_added"] == group]
            if len(item):
                item = item.iloc[0]
                rows.append({"split": "all evaluated masks", "metric": f"{group} Top-1", "PRE": item.get("pre_top1"), "POST": item.get("post_top1")})
    display(pd.DataFrame(rows))
    print("FINAL DECISION:", run_summary.get("decision"))
    print("Critical pass:", run_summary.get("critical_pass"))
    print("Run directory:", RUN_DIR)
    print("=" * 70)


## 11. Artefactos y cierre de la Etapa 3

El directorio MSI contiene configuración resuelta, entorno, resumen de split y chunking, posiciones fijas, predicciones PRE/POST, métricas globales y por token, historia, checkpoint best/latest, movimiento de embeddings, ejemplos y PNGs. Si las verificaciones críticas pasan, el runner guarda modelo y tokenizer en `dapt/models/chemberta_zinc_slices_dapt/` con `dapt_manifest.json`. La siguiente fase (fine-tuning para propiedades) se hará aparte; este cuaderno no crea targets de regresión, splits downstream ni tokenización ChemBERTa para entrenamiento supervisado.